In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,cross_val_score,cross_validate,GridSearchCV,RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score,roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

In [2]:
df = pd.read_csv('train_cleaned.csv')

In [3]:
x = df.iloc[:,:-1]
y = df.iloc[:,-1]
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=230)
scalar = StandardScaler()
x_train1 = scalar.fit_transform(x_train)
x_test1 = scalar.transform(x_test)
x1 = scalar.fit_transform(x)

In [14]:
rm = RandomForestClassifier(max_depth=3,n_jobs=-1)
rm.fit(x_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,3
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [15]:
y_pred = rm.predict(x_test)
y_proba = rm.predict_proba(x_test)[:,1]

In [16]:
print('Accuracy :',np.round(accuracy_score(y_test,y_pred),3))
print('Precision :',np.round(precision_score(y_test,y_pred),3))
print("Recall :",np.round(recall_score(y_test,y_pred),3))
print('f1_score :',np.round(f1_score(y_test,y_pred),3))
print('roc_auc :',np.round(roc_auc_score(y_test,y_proba),3))

Accuracy : 0.87
Precision : 0.874
Recall : 0.829
f1_score : 0.851
roc_auc : 0.943


In [4]:
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights_dict = dict(zip(classes, weights))


criterion = ['gini','entropy','log_loss']
max_depth = [3,5,7,9,11]
max_features = ['sqrt','log2',None,0.5,0.8]
class_weight = ['balanced','balanced_subsample',None]
ccp_alpha = [0.0,0.001,0.01]
# min_sample_split = [2,4,6,8]
# min_sample_leaf = [1,3,5]
# max_leaf_nodes = [2,5,7,None]
accuracy ,precisions , recalls , f1_scores , roc_auc = [],[],[],[],[]
max_depth1 ,n_estimators1 , criterian1 , max_features1 = [],[],[],[]
class_weight1, ccp_alpha1 = [],[]
for criteria in criterion:
    for depth in max_depth:
        for feature in max_features:
            for weight in class_weight:
                for alpha in ccp_alpha:
                    batch_size = 20000
                    rf = RandomForestClassifier(n_estimators=10,
                                               criterion=criteria,
                                               max_depth=depth,
                                               max_features=feature,
                                               class_weight=class_weights_dict,
                                               ccp_alpha=alpha,
                                               n_jobs=-1,
                                               warm_start=True)
                    for i in range(0,x_train.shape[0],batch_size):
                        x_part = x_train.iloc[i:i+batch_size,:]
                        y_part = y_train.iloc[i:i+batch_size]
                        rf.fit(x_part,y_part)
                        rf.n_estimators += 10
                    y_pred = rf.predict(x_test)
                    y_proba = rf.predict_proba(x_test)[:,1]
                    accuracy.append(np.round(accuracy_score(y_test,y_pred),3))
                    precisions.append(np.round(precision_score(y_test,y_pred),3))
                    recalls.append(np.round(recall_score(y_test,y_pred),3))
                    f1_scores.append(np.round(f1_score(y_test,y_pred),3))
                    roc_auc.append(np.round(roc_auc_score(y_test,y_proba),3))
                    max_depth1.append(depth)
                    n_estimators1.append(rf.n_estimators)
                    criterian1.append(criteria)
                    max_features1.append(feature)
                    class_weight1.append(weight)
                    ccp_alpha1.append(alpha)

In [6]:
df = pd.DataFrame({
        'max_depth':max_depth1,
        'criterion':criterian1,
        'max_features':max_features1,
        'class_weight':class_weight1,
        'n_estimators':n_estimators1,
        'ccp_alpha':ccp_alpha1,
        'accuracy':accuracy,
        'precision':precisions,
        'recall':recalls,
        'f1_score':f1_scores,
        'roc_auc':roc_auc,
})

In [7]:
df.to_csv('all_combinations_random_forest.csv',index=False)

In [8]:
df

,max_depth,criterion,max_features,class_weight,n_estimators,ccp_alpha,accuracy,precision,recall,f1_score,roc_auc
0,3,gini,sqrt,balanced,240,0.000,0.875,0.861,0.859,0.860,0.944
1,3,gini,sqrt,balanced,240,0.001,0.873,0.854,0.864,0.859,0.943
2,3,gini,sqrt,balanced,240,0.010,0.870,0.842,0.873,0.857,0.941
3,3,gini,sqrt,balanced_subsample,240,0.000,0.876,0.858,0.867,0.863,0.945
4,3,gini,sqrt,balanced_subsample,240,0.001,0.873,0.853,0.867,0.860,0.944
...,...,...,...,...,...,...,...,...,...,...,...
670,11,log_loss,0.8,balanced_subsample,240,0.001,0.879,0.855,0.879,0.867,0.948
671,11,log_loss,0.8,balanced_subsample,240,0.010,0.855,0.812,0.880,0.845,0.931
672,11,log_loss,0.8,None,240,0.000,0.886,0.871,0.876,0.873,0.953
673,11,log_loss,0.8,None,240,0.001,0.879,0.855,0.878,0.867,0.948


In [5]:
# batch_size = 20000
# clf = RandomForestClassifier(warm_start=True,n_jobs=-1,n_estimators=10)
# for i in range(0,x_train.shape[0],batch_size):
#     x_part = x_train.iloc[i:i+batch_size,:]
#     y_part = y_train.iloc[i:i+batch_size]
#     clf.fit(x_part,y_part)
#     clf.n_estimators += 10

In [8]:
# y_pred = clf.predict(x_test)
# y_proba = clf.predict_proba(x_test)[:,1]
# accuracy_score(y_test,y_pred)
# roc_auc_score(y_test,y_proba)

0.9521424123056649